In [2]:
import requests
import json
import pandas as pd
import copy
import time
import requests
import pandas as pd
import json
import copy
import time

In [3]:
def generar_meses():
    meses = ["ene", "feb", "mar", "abr", "may", "jun",
             "jul", "ago", "sep", "oct", "nov", "dic"]
    
    resultado = []
    for anio in range(2015, 2026):
        for mes in meses:
            resultado.append(f"{mes}. {str(anio)[-2:]}")
    
    return resultado


def parse_looker_response(data, mes):
    dataset = data['dataResponse'][0]['dataSubset'][0]['dataset']['tableDataset']
    
    columns_list = dataset['column']
    total_rows = dataset['size']

    final_columns = {}

    for i, col_data in enumerate(columns_list):

        col_name = col_data.get('name') or f"col_{i}"

        key = 'stringColumn' if 'stringColumn' in col_data else 'doubleColumn'
        values = col_data.get(key, {}).get('values', [])
        null_indices = col_data.get('nullIndex', [])

        full_col = []
        val_ptr = 0

        for idx in range(total_rows):
            if idx in null_indices:
                full_col.append(None)
            else:
                full_col.append(values[val_ptr])
                val_ptr += 1

        final_columns[col_name] = full_col

    df = pd.DataFrame(final_columns)
    df["mes"] = mes

    return df


# 🔥 NUEVA FUNCIÓN
def find_mes_filter(payload):
    filters = payload['dataRequest'][0]['datasetSpec']['filters']
    
    for i, f in enumerate(filters):
        try:
            exp = f['filterDefinition']['filterExpression']
            
            if 'stringValues' in exp:
                val = exp['stringValues'][0]
                
                # detecta formato tipo "ene. 24"
                if isinstance(val, str) and "." in val:
                    return i
        except:
            continue
    
    raise ValueError("No se encontró el filtro de mes")


def run_looker_pipeline(
    cookies, 
    headers, 
    params, 
    json_data,
    pais=None,
    ciudad=None,
    operacion=None,
    columnas=None,
    mes_filter_idx=None,        # 🔥 NUEVO
    mes_field_name=None         # 🔥 OPCIONAL (ej: "_fecha_")
):
    
    meses = generar_meses()
    dfs = []

    # ==============================
    # 🔥 DETECCIÓN DEL FILTRO
    # ==============================
    if mes_filter_idx is None:
        filters = json_data['dataRequest'][0]['datasetSpec']['filters']
        
        for i, f in enumerate(filters):
            try:
                field = f['filterDefinition']['filterExpression']['queryTimeTransformation']['dataTransformation']['sourceFieldName']
                
                if mes_field_name and field == mes_field_name:
                    mes_filter_idx = i
                    break
            except:
                continue

        if mes_filter_idx is None:
            raise ValueError("No se encontró el filtro de mes")

    print(f"📍 Usando filtro de mes en índice: {mes_filter_idx}")

    # ==============================
    # LOOP
    # ==============================
    for mes in meses:
        try:
            payload = copy.deepcopy(json_data)

            payload['dataRequest'][0]['datasetSpec']['filters'][mes_filter_idx]['filterDefinition']['filterExpression']['stringValues'] = [mes]

            response = requests.post(
                'https://lookerstudio.google.com/embed/u/0/batchedDataV2',
                params=params,
                cookies=cookies,
                headers=headers,
                json=payload,
            )

            if response.status_code != 200:
                print(f"❌ Error en {mes}")
                continue

            raw_str = response.text[5:].strip()
            data = json.loads(raw_str)

            df = parse_looker_response(data, mes)

            if len(df) == 0:
                print(f"⚠️ {mes} sin datos")
                continue

            # ==============================
            # METADATOS
            # ==============================
            if pais is not None:
                df["pais"] = pais
            if ciudad is not None:
                df["ciudad"] = ciudad
            if operacion is not None:
                df["operacion"] = operacion

            # ==============================
            # RENOMBRE
            # ==============================
            if columnas is not None:
                df = df.rename(columns=columnas)

            dfs.append(df)

            print(f"✅ {mes} | {len(df)} filas")

            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Error en {mes}: {e}")

    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        return pd.DataFrame()

In [4]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AJz30zLefP-2EqQPfyAeLPgW0p4cA:1774509696438',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCU1tqP2KIR7sREia1cfurcOv1e3OzKW4E_VGxQDPoJ8qzl5mzUPcGJBOyfDZEAA',
    '__Secure-3PSIDCC': 'AKEyXzU5qqcYu1aNxnNnrxd9N9BHHyueue5z1bKd1ZtJqrGmbfiTjSiIyBSWul44W37aWNz7MmaBGA',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/dc156dfa-b12c-4f55-bf14-65b7ae451567/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AJz30zLefP-2EqQPfyAeLPgW0p4cA:1774509696438',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AJz30zLefP-2EqQPfyAeLPgW0p4cA:1774509696438; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-; __Secure-3PSIDTS=sidts-CjEBWhotCU1tqP2KIR7sREia1cfurcOv1e3OzKW4E_VGxQDPoJ8qzl5mzUPcGJBOyfDZEAA; __Secure-3PSIDCC=AKEyXzU5qqcYu1aNxnNnrxd9N9BHHyueue5z1bKd1ZtJqrGmbfiTjSiIyBSWul44W37aWNz7MmaBGA',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': 'dc156dfa-b12c-4f55-bf14-65b7ae451567',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '4f53dde5-ccd3-47a8-a13a-1683958b3094',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_u5wcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_vfpcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_duucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_euucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_duucznzjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_9p89ss7ihc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'QTO',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_mspdznzjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'sep. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_vs1cznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [5]:
quito_venta = run_looker_pipeline(cookies, headers, params, json_data, pais="Ecuador",ciudad="Quito",operacion="sell", 
                                mes_field_name="_fecha_",
                                 
                                 columnas={"col_0":"neighborhood",
                                                                                                                                     "col_1":"new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"
                                                                                                                                     })
print("Total de registros", len(quito_venta))
quito_venta.to_csv("quito_venta.csv", encoding="latin1",index=False)
quito_venta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin dato

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,Cumbayá,1755.0,1658,1553.0,may. 22,Ecuador,Quito,sell
1,Iñaquito,1901.0,1448,1321.0,may. 22,Ecuador,Quito,sell
2,Tumbaco,1480.0,1396,1261.0,may. 22,Ecuador,Quito,sell
3,Nayón,1405.0,1361,1346.0,may. 22,Ecuador,Quito,sell
4,Mariscal Sucre,1648.0,1314,1159.0,may. 22,Ecuador,Quito,sell
...,...,...,...,...,...,...,...,...
804,Chimbacalle,NaN,726,NaN,oct. 25,Ecuador,Quito,sell
805,Quitumbe,NaN,697,668.0,oct. 25,Ecuador,Quito,sell
806,Puengasí,790.0,671,562.0,oct. 25,Ecuador,Quito,sell
807,Chilibulo,NaN,669,669.0,oct. 25,Ecuador,Quito,sell


In [6]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AKIoI3TkIKmWoXoD28DGFcdeek1Tw:1774509899180',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-',
    '__Secure-3PSIDTS': 'sidts-CjEBBj1CYn2M8i_kl5KwJ-qWiE_2fdzdGMqvCgDc9qnd7izCPztGEvJUiKSpVPe7Iqg4EAA',
    '__Secure-3PSIDCC': 'AKEyXzWfObkmRJvrlTmBscgQgB-HFEHNE88fP4pkdCoBIwYButMZ0GAoPq1YLp8iXrzaks0YB-SmUg',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/6cee33c6-0910-48a2-8889-7246f4cae95e/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AKIoI3TkIKmWoXoD28DGFcdeek1Tw:1774509899180',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AKIoI3TkIKmWoXoD28DGFcdeek1Tw:1774509899180; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-; __Secure-3PSIDTS=sidts-CjEBBj1CYn2M8i_kl5KwJ-qWiE_2fdzdGMqvCgDc9qnd7izCPztGEvJUiKSpVPe7Iqg4EAA; __Secure-3PSIDCC=AKEyXzWfObkmRJvrlTmBscgQgB-HFEHNE88fP4pkdCoBIwYButMZ0GAoPq1YLp8iXrzaks0YB-SmUg',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '6cee33c6-0910-48a2-8889-7246f4cae95e',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '8195b1fb-609b-4c7d-afc8-88810deb3e48',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_imb9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_oz68k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_1a98k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_u298k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_1a98k9zjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_6dd35chjhc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'QTO',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_lk69k9zjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'sep. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_b1g9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [7]:
quito_renta = run_looker_pipeline(cookies, headers, params, json_data, pais="Ecuador",ciudad="Quito",operacion="rent", 
                                mes_field_name="_fecha_",
                                 columnas={"col_0":"neighborhood",
                                                                                                                                     "col_1":"new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"
                                                                                                                                     })
print("Total de registros", len(quito_renta))
quito_renta.to_csv("quito_renta.csv", encoding="latin1",index=False)
quito_renta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin dato

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,Cumbayá,747.0,556,501.0,may. 22,Ecuador,Quito,rent
1,Iñaquito,571.0,476,470.0,may. 22,Ecuador,Quito,rent
2,Nayón,NaN,428,435.0,may. 22,Ecuador,Quito,rent
3,Tumbaco,623.0,416,335.0,may. 22,Ecuador,Quito,rent
4,Mariscal Sucre,NaN,403,402.0,may. 22,Ecuador,Quito,rent
...,...,...,...,...,...,...,...,...
480,Ponceano,NaN,300,299.0,oct. 25,Ecuador,Quito,rent
481,El Condado,NaN,295,296.0,oct. 25,Ecuador,Quito,rent
482,Carcelén,NaN,272,272.0,oct. 25,Ecuador,Quito,rent
483,Cotocollao,NaN,251,251.0,oct. 25,Ecuador,Quito,rent
